This notebook reconstructs the orbits of fireball impactors listed in the <a href="https://cneos.jpl.nasa.gov/fireballs/">CNEOS Fireball Database</a> by performing backward numerical integrations using the REBOUND integrator. Given the impact location and velocity, we compute the orbital elements corresponding to each fireball’s orbit.

In [ ]:
import spiceypy as spy
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from utils.Utils import (
    Geo2Rec,
    Geo2Eclip,
    get_velocity_ecliptic,
    mag
    )
import rebound as rb
import numpy as np
from astropy.time import Time
import plotly.graph_objects as go
from plotly.subplots import make_subplots
%load_ext autoreload 
%autoreload 2

In [ ]:
#load the necessary spiceypy kernels

path = '../datos/kernels/'
spy.furnsh([path + 'naif0012.tls', path + 'pck00010.tpc', path + 'earth_fixed.tf', path + 'earth_720101_230601.bpc', path + 'earth_latest_high_prec.bpc'])

In [ ]:
#read the CNEOS fireball database

data_fireballs = pd.read_csv('../datos/cneos_fireballs.csv', comment='#')

## Chelyabinsk bolid backward integration example

Before performing the backward integration for all fireballs in the catalog, we begin by testing and illustrating the integration process using the well-known fireball associated with the Chelyabinsk impact event. By sorting the CNEOS fireball database in descending order by impact energy, we can easily identify the Chelyabinsk impactor, as this historic event currently holds the highest recorded impact energy.

In [ ]:
data_fireballs.sort_values('Total Radiated Energy (J)', ascending=False).head(5)

In [ ]:
#We located the Chelyabinsk bolid (chely) using the index id 37

id_chely = 376
chely = data_fireballs.loc[id_chely]
chely

In [ ]:
#We change the Latitude and Longitude data format in the CNEOS database from string to a float value. 

def change_coord(x):
    if x[-1] == 'N' or x[-1] == 'E':
        new = float(x[:-1])
    elif x[-1] == 'S' or x[-1] == 'W':
        new = -float(x[:-1])
    return new    

date = chely['Peak Brightness Date/Time (UT)']

lon = change_coord(chely['Longitude (deg.)'])
lat = change_coord(chely['Latitude (deg.)'])
alt = chely['Altitude (km)']
vx = chely['vx']
vy = chely['vy']
vz = chely['vz']

print("Chelyabinsk impact data ")
print(f"Date of impact: {date}, Geographic coordinates: {lon, lat}")
print(f"Altitude: {alt}, velocity vector [{vx, vy, vy}]")

In [ ]:
#we get the position vector of the site of impact in rectangular coordinates. 

r = Geo2Rec(lon, lat, alt)  #en km
print("Rectangular coodinates: ", r, mag(r))

In [ ]:
#we get the position vector in ecliptic J2000 coordinares 

r_eclip = Geo2Eclip(lon, lat, alt, date=date, frame='ITRF93') #en km
print("Ecliptic Cooridnates: ", r_eclip, mag(r_eclip))

In [ ]:
#We need trasform observed velocity into the true velocity adding the rotarion velocity of the earth. 
 
v = np.array([vx, vy, vz])  

t_sideral = 86164.09053083288 
w_earth = 2 * np.pi / t_sideral 
omega = np.array([0,0,w_earth]) 

v_E = v + spy.vcrss(omega, r) 

print("True Velocity: ", v_E)

In [ ]:
#Then we get the velocity vector in the Ecliptic J2000 frame

et = spy.utc2et(date)
mx = spy.pxform('ITRF93', 'ECLIPJ2000', et)
v_eclip = spy.mxv(mx, v_E)

print("Ecliptic Velocity: ", v_eclip)

In [ ]:
#start the simulation adding the planets and the bolid

rb.horizons.SSL_CONTEXT = 'unverified'

AU = 149597870 #km
day = 86400

sim = rb.Simulation()
sim.units = 'km', 's', 'kg'
sim.integrator = "IAS15" #define the integrator
sim.dt = -86400  #delta time negative to integrate backwards in time

#We need the time dynamical barycenter to add the position of the planets 
time = Time(date, format="iso")
print("Time TDB: ", time.tdb)
print("Time TDB JD: ", time.tdb.jd)

for i in ["Sun", "199", "299", "399", "499", "599", "699", "799", "899"]:
    sim.add(i, hash=f"{i}", date=f"JD{time.tdb.jd}")

r_earth = np.array(sim.particles["399"].xyz)
v_earth = np.array(sim.particles["399"].vxyz)

#we get the position and the velocity of chely related to the solar system barycenter 
r_asteroid = r_eclip + r_earth 
v_asteroid = v_eclip + v_earth 

print("Position ECLIPJ200: ", r_asteroid/AU)
print("Velocity ECLIPJ200: ", (v_asteroid/AU)*day)

asteroid = sim.add(x=r_asteroid[0], y=r_asteroid[1], z=r_asteroid[2], 
                vx=v_asteroid[0], vy=v_asteroid[1], vz=v_asteroid[2], hash="Chely")

In [ ]:
#verify the status in our simulation

sim.status()

In [ ]:
fig = rb.OrbitPlot(sim)

In [ ]:
hora = 3600
deg = 180/np.pi
AU = 149597870 #km

#using Kepler's third law we calculate the orbital period of chely bolid, given the semi major axis
a = 1.73*AU
mu = sim.particles['Sun'].m*sim.G
periodo_orb = 2*np.pi*np.sqrt(a**3/mu)
print("Chelyabinks bolid orbital period ", periodo_orb)

#we want to integrate backwards in time during four orbital periods 
t_end = 4*periodo_orb
print("Integration time: ", t_end)

In [ ]:
N = 10
times = np.linspace(0, t_end, N)

#integrating the orbit backwards until t_end
orbital_elements = np.zeros((N, 6))
for i,time in enumerate(times):
    sim.integrate(-time)
    sim.move_to_hel()
    chely = sim.particles["Chely"]

    o = chely.orbit()
    orbital_elements[i] = [o.a/AU, o.e, o.inc*deg, o.Omega*deg, o.omega*deg, o.f]

In [ ]:
#Let’s see how the orbital elements change during the backward integration and check if they converge to the values reported in the literature.
for element in orbital_elements:
    print(f"Orbital elements in time: {time} s")   
    print(f"a={element[0]}, e={element[1]}, i={element[2]}, Omega={np.mod(element[3], 360)}, omega={element[4]}, f={element[5]}") 

We observe that our set of orbital elements converges to the values reported in the literature for the Chelyabinsk bolide after about four orbital periods of backward integration. Now, let’s continue by integrating the orbits of all fireballs in the CNEOS database.

## CNEOS Bolids Orbital Elements

We aim to compute the orbital elements of the fireballs listed in the CNEOS catalog by performing backward integrations over four orbital periods. To do this, we first estimate each bolide’s pre-impact orbit in order to calculate its semi-major axis and corresponding orbital period. Then, we carry out the backward integration using the REBOUND N-body integrator, including the gravitational influence of the Sun and the planets.

In [ ]:
fireballs = data_fireballs[["Peak Brightness Date/Time (UT)", "Latitude (deg.)", "Longitude (deg.)", "Altitude (km)", "vx", "vy", "vz"]].dropna().reset_index()
fireballs

In [ ]:
#we change the format of the geographical coordinates latitud and longitud to numerical values

fireballs['lat'] = fireballs["Latitude (deg.)"].apply(change_coord)
fireballs['lon'] = fireballs["Longitude (deg.)"].apply(change_coord)
fireballs[['lat','lon']]


In [ ]:
#get the position and velocity of the bolids in the ECLIPTIC J2000 frame

r_eclip = np.zeros((len(fireballs), 3))
v_eclip = np.zeros((len(fireballs), 3))
for i in range(len(fireballs)):
    date = fireballs["Peak Brightness Date/Time (UT)"][i]
    lat = fireballs["lat"][i]
    lon = fireballs["lon"][i]
    alt = fireballs["Altitude (km)"][i]
    vx = fireballs['vx'][i]
    vy = fireballs['vy'][i]
    vz = fireballs['vz'][i]

    r_eclip[i] = Geo2Eclip(lon, lat, alt, date=date, frame='ITRF93') 
    v_eclip[i] = get_velocity_ecliptic(vx, vy, vz, lon, lat, alt, date=date)

In [ ]:
#get the semimajor axis of each orbit to compute the orbital period. Run this cell may took a while!

#rb.horizons.SSL_CONTEXT = 'unverified'

semimajor_axis = []  
for i in range(len(fireballs)): 
    rb.horizons.SSL_CONTEXT = 'unverified'   
    sim = rb.Simulation()
    sim.units = 'km', 's', 'kg'
    sim.integrator = "IAS15"
    sim.dt = -86400

    date = fireballs["Peak Brightness Date/Time (UT)"][i]
    time = Time(date, format="iso")

    for body in ["Sun", "199", "299", "399", "499", "599", "699", "799", "899"]:
        sim.add(body, hash=f"{body}", date=f"JD{time.tdb.jd}")

    r_earth = np.array(sim.particles["399"].xyz)
    v_earth = np.array(sim.particles["399"].vxyz)

    r_asteroid = r_eclip[i] + r_earth 
    v_asteroid = v_eclip[i] + v_earth  

    asteroid = sim.add(x=r_asteroid[0], y=r_asteroid[1], z=r_asteroid[2], 
                    vx=v_asteroid[0], vy=v_asteroid[1], vz=v_asteroid[2], hash="Asteroid")
    
    asteroid = sim.particles["Asteroid"]
    orbit = asteroid.orbit()
    
    semimajor_axis.append({'index': i, 'a': orbit.a})

In [ ]:
semimajor_axis

We notice that some semi-major axis values are negative, indicating non-Keplerian, hyperbolic orbits. In this study, we focus only on elliptical orbits objects that complete full orbital periods.

In [ ]:
orbital_periods = []  #save calculated orbital periods 
mu = sim.particles["Sun"].m*sim.G
for orbit in semimajor_axis:    
    if orbit['a'] < 0: 
        #is hiperbolic orbit
        continue 
    else:
        periodo_orb = 2*np.pi*np.sqrt(orbit['a']**3/mu)
        t_end = 4*periodo_orb 
        orbital_periods.append({'index': orbit['index'], 'time': t_end})

In [ ]:
# The code below calculates the integration time needed for each orbit to complete four orbital periods.
# Let’s now compute the orbital elements by integrating the motion backward in time. 
# This code also takes a while...

rb.horizons.SSL_CONTEXT = 'unverified'
orbit_elements = np.zeros((len(orbital_periods), 6))

for i, period in enumerate(orbital_periods):
    index = period['index']
    int_time = period['time']
    
    sim = rb.Simulation()
    sim.units = 'km', 's', 'kg'
    sim.integrator = "IAS15"
    sim.dt = -86400

    date = fireballs["Peak Brightness Date/Time (UT)"][index]
    time = Time(date, format="iso")

    for body in ["Sun", "199", "299", "399", "499", "599", "699", "799", "899"]:
        sim.add(body, hash=f"{body}", date=f"JD{time.tdb.jd}")

    r_earth = np.array(sim.particles["399"].xyz)
    v_earth = np.array(sim.particles["399"].vxyz)

    r_asteroid = r_eclip[index] + r_earth 
    v_asteroid = v_eclip[index] + v_earth #así si es 

    asteroid = sim.add(x=r_asteroid[0], y=r_asteroid[1], z=r_asteroid[2], 
                    vx=v_asteroid[0], vy=v_asteroid[1], vz=v_asteroid[2], hash="Asteroid")

    sim.integrate(-int_time)
    sim.move_to_hel()

    asteroid = sim.particles["Asteroid"]
    orbit = asteroid.orbit()

    orbit_elements[i] = [orbit.a, orbit.e, orbit.inc, orbit.Omega, orbit.omega, orbit.M]

In [ ]:
orbit_elements

### Save orbital elements 

In [ ]:
fireballs_int = pd.DataFrame()
fireballs_int['a (km)'] = orbit_elements[:,0]
fireballs_int['e'] = orbit_elements[:,1]
fireballs_int['i (rad)'] = orbit_elements[:,2]
fireballs_int['Omega (rad)'] = orbit_elements[:,3]
fireballs_int['omega (rad)'] = orbit_elements[:,4]
fireballs_int['M (rad)'] = orbit_elements[:,5]

fireballs_int.to_csv("data/orbital_elements_integration.csv", index=False)

### Read the saved orbital elements and plot it

In [30]:
cneos_int = pd.read_csv("../datos/orbital_elements_integration.csv")

#we want to plot the impactors orbital elements results agains the orbital elements of the NEOs
neos = pd.read_csv("../datos/sbdb_query_results_NEOS.csv")
neos["q"] = neos["a"]*(1-neos["e"])
neos = neos[neos['q'] < 1.30]

Let's visualize the orbital elements resulting from the backward integration.

In [32]:
AU = 149597870 #km
deg = 180/np.pi

cneos_int["a"] = cneos_int["a (km)"]/AU
cneos_int["q"] = cneos_int["a"]*(1-cneos_int["e"])
cneos_int["i"] = cneos_int["i (rad)"]*deg
cneos_int['om'] = np.mod(cneos_int['Omega (rad)']*deg, 360)
cneos_int['w'] = np.mod(cneos_int['omega (rad)']*deg, 360)
cneos_int['ma'] = np.mod(cneos_int['M (rad)']*deg, 360)
cneos_int = cneos_int[cneos_int["q"] < 1.30]

In [33]:
q_cneos = np.array(cneos_int['q'])
q_neos = np.array(neos['q'])
e_cneos = np.array(cneos_int['e'])
e_neos = np.array(neos['e'])
i_cneos = np.array(cneos_int['i'])
i_neos = np.array(neos['i'])


fig = go.Figure()

# NEOS
fig.add_trace(go.Splom(
    dimensions=[
        dict(label='q [au]', values=q_neos),
        dict(label='e', values=e_neos),
        dict(label='i [°]', values=i_neos)
    ],
    name="NEOs",
    showupperhalf=False,
    diagonal_visible=False,
    marker=dict(color="#6baed6", size=1, opacity=0.3)
))

# Impactadores
fig.add_trace(go.Splom(
    dimensions=[
        dict(label='q [au]', values=q_cneos),
        dict(label='e', values=e_cneos),
        dict(label='i [°]', values=i_cneos)
    ],
    name="Earth Impactors",
    showupperhalf=False,
    diagonal_visible=False,
    marker=dict(color="#ED5C5E", size=3, opacity=0.9)
))

# Layout minimalista
fig.update_layout(
    title_x=0.5,
    width=650,
    height=650,
    font=dict(family="Arial", color="#444444", size=14),
    plot_bgcolor='white',
    paper_bgcolor='white',
    dragmode='select',
    hovermode='closest',
    legend=dict(
        x=0.8, y=1.05,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0
    )
)

# Ajuste de ejes para estilo limpio
for axis in ['xaxis', 'xaxis2', 'xaxis3', 'yaxis', 'yaxis2', 'yaxis3']:
    fig.update_layout({axis: dict(
        showline=True,
        linewidth=1,
        linecolor="#333333",
        mirror=True,
        showgrid=True,
        gridcolor="#e6e6e6",
        zeroline=False
    )})

fig.show()